In [31]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
import time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4
max_iters = 1000
#eval_interval = 2500
learning_rate = 3e-4
eval_iters = 250

#Dropout drops random neurons so we dont overfit, drops 20% of neurons at random
#It gets disabled on eval
#Enabled during training
#dropout = 0.2

cpu


In [32]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
print(chars)
vocab_size = len(chars)

['\n', ' ', '!', '"', '#', '$', '%', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', ']', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '—', '‘', '’', '“', '”', '•', '™']


In [33]:
string_to_int = {ch:i for i, ch in enumerate(chars) }
int_to_string = {i:ch for i, ch in enumerate(chars) }

#Initialize Encoder and Decoder
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72, 60,
        63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73, 78,
        66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,  1,
        67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63, 31,
        73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77, 63,
         1, 73, 64,  1, 59, 72, 83, 73, 72, 63])


In [34]:
#Get training values
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1]for i in ix])
    #Push to the GPU
    x,y = x.to(device), y.to(device)
    return x,y

x,y = get_batch('train')
print('inputs:')

#Print(x.shape)
print(x)
print('targets:')
print(y)


inputs:
tensor([[59, 14, 64, 76, 79, 67, 78,  1],
        [62,  1, 81, 59, 77,  1, 61, 59],
        [59, 71,  0, 78, 66, 63,  1, 52],
        [78, 70, 83, 13,  1,  3, 67, 78]])
targets:
tensor([[14, 64, 76, 79, 67, 78,  1, 65],
        [ 1, 81, 59, 77,  1, 61, 59, 76],
        [71,  0, 78, 66, 63,  1, 52, 67],
        [70, 83, 13,  1,  3, 67, 78,  1]])


In [35]:
#For training there are no gradients
@torch.no_grad()
def estimate_loss():
    out = {}
    #eval used when model is being evaluated or tested
    #dropout and batch normalization behave differently in this mode
    #We want everything to be working together and how well it performs
    #Get network in it's optimal form and see how good the results are
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    #Training mode is when we try to throw as much to the network as we can for it to learn
    #It challenges the network
    model.train()
    return out
        


x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, 'target is', target)

when input is tensor([49]) target is tensor(66)
when input is tensor([49, 66]) target is tensor(63)
when input is tensor([49, 66, 63]) target is tensor(1)
when input is tensor([49, 66, 63,  1]) target is tensor(45)
when input is tensor([49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([49, 66, 63,  1, 45, 76, 73]) target is tensor(68)
when input is tensor([49, 66, 63,  1, 45, 76, 73, 68]) target is tensor(63)


In [36]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embeddings_table = nn.Embedding(vocab_size, vocab_size)
        #This is basically a lookup table of the token and the scores to
        #what follows it

        #An attention layer would be utilized with:
        #nn.Embedding(vocab_size, n_embd) plus a lm_head

    def forward(self, index, targets=None):
        #This is a row lookup
        #Index is (B,T) of integers.
        #Each integer gets replaced by its row. Output is (B,T,C) where C = vocab_size.
        #For each B sequences, for each of T positions, get a full score vector over the vocab
        logits = self.token_embeddings_table(index)
        

        if targets is None:
            loss = None
        else:
            #F.cross_entropy wants predictions as (N,C) and targets as (N,)
            #A flat list of N independent predictions
            #Currently we have a 3D block, so we need to collapse batch and time into one axis
            #In this case it is: B*T separate predictions each over C classes.
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            #get the predictions
            logits, loss = self.forward(index)

            #Focus only on the last time step
            logits = logits[:,-1,:] #becomes (B,C)

            #apply softmax to get the probability distribution
            #dim=-1 normalizes across the C axis so each row sums to 1
            #And becomes a valid probability distribution
            #If it was dim=0 it'd normalize across the batch.
            probs = F.softmax(logits, dim=-1) # (B,C)

            #sample from the distribution
            #torch.multinomial samples rather than taking argmax.
            #Argmax would be deterministic and collapse into loops
            #Sampling makes the output varied
            index_next = torch.multinomial(probs, num_samples=1) # (B, 1)

            #append sampled index to the running sequence
            index = torch.cat((index, index_next), dim=1) # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size)
m = model.to(device)

context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)
    


PI#)gpbJ+fBJ$uZ6_4af
aokE#LQASIY”%bRK6+DR-n!a!'D135cJ:%ga’b9Gra.xlS#:DmX+QGNh..hC0/d+“9b
eR kvA•5WiwSn+uA+:%“H
RMzMw/Ygq3NrH(o]s]qH( xV'8%7a•D_*R.p 3!SfkA8LK_VmC!’’,C._w!NPo%-“TbU‘7’GH“m:%CTMc'0Y”xiV!o“1JDn(m?;xz+”5ygaRVK3—?#LZ%80xi P%/d”’Laf'JwN(—-pV3,e-31U'sH”PX_j5—?Aj;lp8Chjq/qU‘h™q’w?sU;—a&d06pE311[B—FwC9H3D*bAd“Q9QAp[s(5%QLe‘3:“•w/[L“#-[_5U”5WVE;dZht”sUA"‘h8v/!0%r“dT[BBDvkrCFv—X
r?vz)M.Pppf••qBQ_HH3!BH
"uAlV'JVNq7X'”z_A‘&3/6:8wf.ag$L(8J$“#%tp22Lp]"Po8TqWhOE+H]IVlm;”P—,L5,0—'4/0ccbF8K?gt•rv7


In [37]:

#Create PyTorch Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):

    if iter % eval_iters == 0:
        losses = estimate_loss()
        print(f'step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}')

    #Sample a batch of data
    xb, yb = get_batch('train')

    #Evaluate the loss
    logits, loss = model.forward(xb,yb)
    #Set gradient to None instead of 0 because None occupies less space
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

step: 0, train loss: 4.929, val loss: 4.942
step: 250, train loss: 4.851, val loss: 4.886
step: 500, train loss: 4.816, val loss: 4.816
step: 750, train loss: 4.745, val loss: 4.762
4.518755912780762


In [38]:
context = torch.zeros((1,1), dtype=torch.long, device=device)
generated_chars = decode(m.generate(context, max_new_tokens=500)[0].tolist())
print(generated_chars)


srEO9aRL™#[q33/v/dPQ—8+V&nSv:LzRs,
:-g%lVL("T0K“+AeR/A6'6'9q—yU#j&lJ+VM5UUN2_5v'FsNAc8u3;q"H k1WonWJ:E•;l9q“

*:%15N($k,‘*.D/maA•FC[(8-l)H(SN2(MH34(OJ$m4
m9"wEk#*r”xEse+CFDZx50/[f8Vf
•Dr6cUS?A:™+c’PP0f:t!'DO(j)?#$YFIJ+
’Wiv*yU5WfS#9GO5pj#”x*p’7eN"SigeH6_a”Rv;Enyc.”_2)’*‘h,
z;Sz!T™+R“!k,(5YK:DgjF“KubM“9pKGe,EX1D+5kdBGxh47coTaNEYyC_Mz3Xq
”_p ;4E#5OovG“K7.a•oT0K“9GH35Yd+H(vm-U.&
9_ht-Xp MbmrBaf5)CX)RVe9HSA”_.(q"XH52o ”_0QEtro•p*nZkVuwH!]3gx”_&E"Wuw0kASzMtvA"Kt—n“O‘
Dc’PIX*JDv/q333Wrp6wtpp!Tm*W5q)??
